# MuCoCo RQ 1 Experiment Results Aggregation

This notebook is used to aggregate the results for MuCoCo RQ1 experiments. The results are stored in MuCoCo_results/MuCoCo_experiment_results/ in the project root folder. The final aggregated results from this notebook are used in tables VI (aggregating across model), VII (aggregating across tasks) and VIII (aggregating across benchmarks). 

In [16]:
import os
import sys
import pandas as pd
from typing import Tuple, Dict

In [17]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [18]:
from utility.data_log_functions import DataLogHelper

In [42]:
import subprocess
from pathlib import Path

COMPARE_SCRIPT = "RQ1_BigCodeBench_results_aggregation.py"
PYTHON = os.environ.get("PYTHON", "python")

curr_dir = Path.cwd()
par_dir = curr_dir.parent
task_dir = par_dir / "test_res" / "code_generation"

model_names = [m for m in os.listdir(task_dir) if m != ".DS_Store"]
repo_root = curr_dir.parents[1]

env = os.environ.copy()
env["PYTHONPATH"] = str(repo_root) + os.pathsep + env.get("PYTHONPATH", "")

# Use absolute script path (safer)
script_path = curr_dir / COMPARE_SCRIPT
processes = {}
bars = {}

bigcodebench_res_dir = curr_dir / "bigcodebench_json"
# bigcodebench_res_dir = Path("/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/MuCoCo_results/MuCoCo_experiment_results/code_generation/bigcodebench_json")
os.makedirs(bigcodebench_res_dir, exist_ok=True)

In [20]:
# for idx, model in enumerate(model_names):
#     out_json = bigcodebench_res_dir / f"{model}_bigcodebench_results.json"
#     res_dir = task_dir / model

#     p = subprocess.Popen(
#         [
#             PYTHON,
#             COMPARE_SCRIPT,
#             "--res_dir", str(res_dir),
#             "--out_path", str(out_json),
#         ],
#         cwd = str(curr_dir),
#         env = env,
#         stdout=subprocess.PIPE,
#         stderr=subprocess.PIPE,
#         text=True,
#     )
#     processes[p] = model

# # Wait for all
# for p, model in processes.items():
#     stdout, stderr = p.communicate()
#     if p.returncode != 0:
#         print(f"[ERROR] {res_dir}")
#         print(stderr[-2000:])
#     else:
#         print(f"[OK] {res_dir}")
    


## Parsing the BigCodeBench Results

In [43]:
model_dict = {
    "Qwen2.5-Coder-14B-Instruct" : "Qwen2.5-Coder-14B-Instruct",
    "gemma-3-12b-it": "Gemma-3-12b-it",
    "deepseek-reasoner": "DeepSeek-V3.2-Exp (Non-thinking Mode)",
    "LLama-3.1-8B": "LLama-3.1-8B",
    "gpt-5" : "GPT-5",
    "gpt-4o": "GPT-4o",
    "codestral-latest": "codestral-2508",
}

In [44]:
import json
bigcodebench_res = {}
for json_file in os.listdir(bigcodebench_res_dir):
    res_dir = bigcodebench_res_dir / json_file
    with open(res_dir, 'r') as file:
        data = json.load(file)
    model_name = model_dict[json_file.split('_')[0]]
    bigcodebench_res[model_name] = data
    print(data)


{'Random': {'total_inconsistencies': 234, 'total_questions': 1110, 'total_success': 853, 'total_answered': 1110, 'cumulative_inconsistency_distance': 165.32548562548567}, 'Sequential': {'total_inconsistencies': 227, 'total_questions': 1110, 'total_success': 871, 'total_answered': 1112, 'cumulative_inconsistency_distance': 159.3921522921523}, 'Lexical': {'total_inconsistencies': 461, 'total_questions': 2220, 'total_success': 1724, 'total_answered': 2222, 'cumulative_inconsistency_distance': 324.71763791763794}}
{'Random': {'total_inconsistencies': 219, 'total_questions': 1110, 'total_success': 851, 'total_answered': 1111, 'cumulative_inconsistency_distance': 154.16386668886668}, 'Sequential': {'total_inconsistencies': 216, 'total_questions': 1111, 'total_success': 853, 'total_answered': 1112, 'cumulative_inconsistency_distance': 149.7781524031524}, 'Lexical': {'total_inconsistencies': 435, 'total_questions': 2221, 'total_success': 1704, 'total_answered': 2223, 'cumulative_inconsistency_

In [23]:
def compare_logs_against_no_mutation(res_dir: str, task: str, benchmark: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = (), ):
    """
    This function compares all LLM output logs located in res_dir with the no_mutation LLM output log.

    Args:
        res_dir: directory to the csv files. this directory should also contain a "no_mutation" log output file
        task: the task type (e.g.: code generation, input prediction, etc)
        filter: strings that should be inside the log names of the csv log outputs
        anti-filter: strings that should NOT be inside the log names of the csv log outputs

    Returns:
        results_df: Pandas Dataframe containing inconsistency scores 
        category_dict: Python Dictionary containing aggegated scores
    """
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    csv_logs.sort()
    target_log_name = [l for l in csv_logs if "no_mutation" in l][-1]
    csv_logs.pop(csv_logs.index(target_log_name))
    target_log_path = os.path.join(res_dir, target_log_name)
    target_log = pd.read_csv(target_log_path)

    results_df = pd.DataFrame()

    total_inconsistencies = 0
    total_questions = 0
    total_success = 0
    total_answered = 0

    cumulative_inconsistency_distance = 0

    category_dict = {}
    mutation_dict = {}

    for log_name in csv_logs:
        # print(log_name)
        log_category = DataLogHelper.obtain_category(log_name)
                
        log2_file_path = os.path.join(res_dir, log_name)
        log2 = pd.read_csv(log2_file_path) 

        target_log, log2 = DataLogHelper.standardize_two_df(target_log, log2)

        inconsistency_dict = DataLogHelper.compare_code_generation_dataframe_results(log1=target_log, log2=log2, task = task, benchmark = benchmark)
        
        # Adding results into the dataframe
        cleaned_mutation_name = DataLogHelper.clean_up_csv_name(log_name.replace('.csv', ''))
        results_df.loc[cleaned_mutation_name, "Inconsistency Score"] = f"{inconsistency_dict['total_inconsistencies']}/{inconsistency_dict['total_inconsistency_comparisons']} ({round((inconsistency_dict['total_inconsistencies'])*100/inconsistency_dict['total_inconsistency_comparisons'], 2)}%)"
        results_df.loc['No Mutation', "Inconsistency Score"] = "N/A"
        results_df.loc['No Mutation', "Model Accuracy"] = f"{(inconsistency_dict['log1_success'])}/{inconsistency_dict['log1_total_answered']} ({round((inconsistency_dict['log1_success'])*100/inconsistency_dict['log1_total_answered'], 2)}%)"
        results_df.loc[cleaned_mutation_name, "Model Accuracy"] = f"{(inconsistency_dict['log2_success'])}/{inconsistency_dict['log2_total_answered']} ({round((inconsistency_dict['log2_success'])*100/inconsistency_dict['log2_total_answered'], 2)}%)"

        if total_success == 0:
            total_success += inconsistency_dict['log1_success']
        
        if total_answered == 0:
            total_answered += inconsistency_dict['log1_total_answered']

        if 'model_ensemble' in log_name.lower() or "ensemble" not in log_name.lower() :
            total_inconsistencies += inconsistency_dict['total_inconsistencies']
            total_questions += inconsistency_dict['total_inconsistency_comparisons']
            total_success += inconsistency_dict['log2_success']
            total_answered += inconsistency_dict['log2_total_answered']
            cumulative_inconsistency_distance += inconsistency_dict['cumulative_inconsistency_distance']
        
        if log_category:
            d: Dict = category_dict.get(log_category, {})
            d['total_inconsistencies'] = d.get('total_inconsistencies', 0) + inconsistency_dict['total_inconsistencies']
            d['total_questions'] = d.get('total_questions', 0) + inconsistency_dict['total_inconsistency_comparisons']
            d['total_success'] = d.get('total_success', 0) + inconsistency_dict['log2_success']
            d['total_answered'] = d.get('total_answered', 0) + inconsistency_dict['log2_total_answered']
            d['cumulative_inconsistency_distance'] = d.get('cumulative_inconsistency_distance', 0) + inconsistency_dict['cumulative_inconsistency_distance']
            category_dict[log_category] = d

        # adding results in mutation_dict, with the mutation name as key
        mutation_dict[cleaned_mutation_name] = {
            'total_inconsistencies': inconsistency_dict['total_inconsistencies'],
            'total_questions': inconsistency_dict['total_inconsistency_comparisons'],
            'total_success': inconsistency_dict['log2_success'],
            'total_answered': inconsistency_dict['log2_total_answered'],
            'cumulative_inconsistency_distance': inconsistency_dict['cumulative_inconsistency_distance']
        }
    
    results_df = pd.concat([
        results_df[results_df.index.str.lower().str.contains("no mutation")],

        results_df[
            ~results_df.index.str.lower().str.contains("ensemble") &
            ~results_df.index.str.lower().str.contains("no mutation")
        ],

        results_df[results_df.index.str.lower().str.contains("ensemble")]
    ])

    ## Adding aggregated second order results and atomic results
    for key, mut_dict in category_dict.items():
        mut_inconsistencies = mut_dict['total_inconsistencies']
        mut_questions = mut_dict['total_questions']
        mut_success = mut_dict['total_success']
        mut_answered = mut_dict['total_answered']
        cum_inc_dist = mut_dict['cumulative_inconsistency_distance']

        results_df.loc[f"{key} Results", "Inconsistency Score"] = f"{mut_inconsistencies}/{mut_answered} ({round(mut_inconsistencies*100/mut_answered, 2)}%)"
        results_df.loc[f"{key} Results", "Inconsistency Distance"] = f"{cum_inc_dist}/{mut_questions} ({round(cum_inc_dist*100/mut_questions, 2)}%)"
        results_df.loc[f"{key} Results", "Model Accuracy"] = f"{mut_success}/{mut_answered} ({round(mut_success*100/mut_answered, 2)}%)"

        mutation_dict[f"{key} Results"] = mut_dict

    results_df.loc["Aggregated Results", "Inconsistency Score"] = f"{total_inconsistencies}/{total_questions} ({round(total_inconsistencies*100/total_questions, 2)}%)"
    results_df.loc["Aggregated Results", "Inconsistency Distance"] = f"{cumulative_inconsistency_distance}/{total_questions} ({round(cumulative_inconsistency_distance*100/total_questions, 2)}%)"
    results_df.loc["Aggregated Results", "Model Accuracy"] = f"{total_success}/{total_answered} ({round(total_success*100/total_answered, 2)}%)"
    
    return [
        results_df, 
        category_dict, 
        ]

In [24]:
current_dir = os.getcwd()
proj_dir = os.path.abspath(os.path.join(current_dir, ".."))

def obtain_benchmark_task_csv(benchmark: str, task: str) -> pd.DataFrame:

    final_df = pd.DataFrame()  # start with an empty DataFrame
    final_dict = {}

    # Iterating through each model in model_dict
    for k, m in model_dict.items():
        # print(k)
        res_dir = os.path.join(proj_dir, f"MuCoCo_experiment_results/{task}/{k}")
        res_dir = os.path.join(proj_dir, f"test_res/{task}/{k}")
        try:
            res, category_dict = compare_logs_against_no_mutation(res_dir=res_dir, filter=(benchmark, ), task = task, benchmark=benchmark)

        except FileNotFoundError:
            print(f"{res_dir} does not exist.")
            continue

        res_df = pd.DataFrame(res)

        res_df = res_df.add_prefix(f"{m} ")

        if final_df.empty:
            final_df = res_df
        else:
            final_df = pd.concat([final_df, res_df], axis=1)

        final_dict[m] = category_dict
    return final_df, final_dict


In [25]:
from tqdm import tqdm
import copy


tasks = {
    'mcq_inconsistency': ['CodeMMLU'],
    'input_prediction': ['HumanEval', "CruxEval"],
    'output_prediction': ['HumanEval', "CruxEval"],
    # 'code_generation': ['BigCodeBench',],
    'code_generation': ["HumanEval",],
}

task_dict = {}
overall_dict = {}
all_benchmark_dict = {}
dfs = []

def combine_two_dictionaries(d1: dict, d2: dict) -> dict:
    out = copy.deepcopy(d1)
    for k, inner2 in d2.items():
        if k not in out:
            out[k] = copy.deepcopy(inner2)              
        else:
            for kk, vv in inner2.items():
                out[k][kk] = out[k].get(kk, 0) + vv
    return out


for task, benchmarks in tqdm(tasks.items()):
    # Dictionary for storing results to aggregate by task
    task_d = {}

    print(f"Aggregating for {task} logs")
    for benchmark in benchmarks:

        print(f"Working on {benchmark} now...")
        final_df, aggregated_dict = obtain_benchmark_task_csv(benchmark, task)

        benchmark_dict = {}

        for model, mut_cat_dict in aggregated_dict.items():
            if "ensemble" in model:
                continue

            for mut_cat, res_dir in mut_cat_dict.items():
                # make a NEW dict here instead of aliasing res_dir
                if not benchmark_dict.get(mut_cat, None):
                    benchmark_dict[mut_cat] = res_dir.copy()
                else:
                    for key, val in res_dir.items():
                        benchmark_dict[mut_cat][key] += val

            d1 = task_d.get(task, {})
            if not d1:
                task_d[task] = copy.deepcopy(mut_cat_dict)
            else:
                task_d[task] = combine_two_dictionaries(d1, mut_cat_dict)
                
        # building benchmark dict for aggregating results by benchmark
        d = all_benchmark_dict.get(benchmark, {})
        if not d:
            all_benchmark_dict[benchmark] = benchmark_dict
        else:
            new_d =  combine_two_dictionaries(d, benchmark_dict) 
            all_benchmark_dict[benchmark] = new_d

        # building overall dictionary for aggregating results by models
        if not overall_dict:
            overall_dict = aggregated_dict
        else:
            for model_name, dict1 in overall_dict.items():
                dict2 = aggregated_dict[model_name]
                overall_dict[model_name] = combine_two_dictionaries(dict1, dict2)
            
    
    task_dict[task] = task_d[task]

    print(task_dict[task])


  0%|          | 0/4 [00:00<?, ?it/s]

Aggregating for mcq_inconsistency logs
Working on CodeMMLU now...


 25%|██▌       | 1/4 [00:03<00:09,  3.07s/it]

{'Logical': {'total_inconsistencies': 743, 'total_questions': 1930, 'total_success': 905, 'total_answered': 1947, 'cumulative_inconsistency_distance': 0}, 'Syntactic': {'total_inconsistencies': 139, 'total_questions': 820, 'total_success': 556, 'total_answered': 820, 'cumulative_inconsistency_distance': 0}, 'Lexical': {'total_inconsistencies': 218, 'total_questions': 2172, 'total_success': 1702, 'total_answered': 2172, 'cumulative_inconsistency_distance': 0}}
Aggregating for input_prediction logs
Working on HumanEval now...
Working on CruxEval now...


 50%|█████     | 2/4 [00:46<00:53, 26.52s/it]

{'Logical': {'total_inconsistencies': 1526, 'total_questions': 21539, 'total_success': 16342, 'total_answered': 21532, 'cumulative_inconsistency_distance': 0}, 'Syntactic': {'total_inconsistencies': 661, 'total_questions': 11200, 'total_success': 9017, 'total_answered': 11197, 'cumulative_inconsistency_distance': 0}, 'Lexical': {'total_inconsistencies': 1991, 'total_questions': 30086, 'total_success': 24242, 'total_answered': 30082, 'cumulative_inconsistency_distance': 0}}
Aggregating for output_prediction logs
Working on HumanEval now...
Working on CruxEval now...


 75%|███████▌  | 3/4 [01:25<00:32, 32.30s/it]

{'Logical': {'total_inconsistencies': 3990, 'total_questions': 21539, 'total_success': 13808, 'total_answered': 21521, 'cumulative_inconsistency_distance': 0}, 'Syntactic': {'total_inconsistencies': 1891, 'total_questions': 11200, 'total_success': 7039, 'total_answered': 11192, 'cumulative_inconsistency_distance': 0}, 'Lexical': {'total_inconsistencies': 6051, 'total_questions': 30086, 'total_success': 18595, 'total_answered': 30071, 'cumulative_inconsistency_distance': 0}}
Aggregating for code_generation logs
Working on HumanEval now...


100%|██████████| 4/4 [02:22<00:00, 35.62s/it]

-0.5
1.0
{'Lexical': {'total_inconsistencies': 527, 'total_questions': 1876, 'total_success': 1187, 'total_answered': 1870, 'cumulative_inconsistency_distance': 316.7169361194361}}


# Aggregated MuCoCo Results Aggregated Across Benchmarks (Table VIII)

In [45]:
# Declaring some BCB lexical variables
bigcodebench_lexical_total_inc = sum([d['Lexical']['total_inconsistencies'] for d in bigcodebench_res.values()])
bigcodebench_lexical_total_qns = sum([d['Lexical']['total_questions'] for d in bigcodebench_res.values()])
bigcodebench_lexical_total_success = sum([d['Lexical']['total_success'] for d in bigcodebench_res.values()])
bigcodebench_lexical_total_answered = sum([d['Lexical']['total_answered'] for d in bigcodebench_res.values()])
bigcodebench_lexical_total_inc_dist = round(sum([d['Lexical']['cumulative_inconsistency_distance'] for d in bigcodebench_res.values()]), 2)

In [46]:
import copy
import json

benchmark_df = pd.DataFrame()
all_cat_dict = {}


for benchmark, mut_cat_dict in all_benchmark_dict.items():
    benchmark_inconsistencies = 0
    benchmark_questions = 0
    benchmark_success = 0
    benchmark_answered = 0
    benchmark_cum_inc_dist = 0

    for mut_cat, res_dir in mut_cat_dict.items():
        if not res_dir:
            continue

        # Make a defensive copy so we don’t mutate shared references
        res_dir = copy.deepcopy(res_dir)

        mut_inconsistencies = res_dir['total_inconsistencies']
        mut_questions = res_dir['total_questions']
        mut_success = res_dir['total_success']
        mut_answered = res_dir['total_answered']

        benchmark_df.loc[mut_cat, f"{benchmark} Inconsistencies"] = f"{mut_inconsistencies}/{mut_questions} ({round(mut_inconsistencies*100/mut_questions, 2)}%)"
        benchmark_df.loc[mut_cat, f"{benchmark} Accuracy"] = f"{mut_success}/{mut_answered} ({round(mut_success*100/mut_answered, 2)}%)"

        benchmark_inconsistencies += mut_inconsistencies
        benchmark_questions += mut_questions
        benchmark_success += mut_success
        benchmark_answered += mut_answered

        d = all_cat_dict.get(mut_cat, {})
        if not d:
            all_cat_dict[mut_cat] = copy.deepcopy(res_dir)
        else:
            for key, value in d.items():
                d[key] += res_dir[key]
            all_cat_dict[mut_cat] = d


    benchmark_df.loc["Aggregated Results", f"{benchmark} Inconsistencies"] = f"{benchmark_inconsistencies}/{benchmark_questions} ({round(benchmark_inconsistencies*100/benchmark_questions, 2)}%)"
    benchmark_df.loc["Aggregated Results", f"{benchmark} Accuracy"] = f"{benchmark_success}/{benchmark_answered} ({round(benchmark_success*100/benchmark_answered, 2)}%)"

# Aggregating BCB results #

benchmark_df.loc["Lexical", "BigCodeBench Inconsistencies"] = f"{bigcodebench_lexical_total_inc}/{bigcodebench_lexical_total_qns} ({round(bigcodebench_lexical_total_inc*100/bigcodebench_lexical_total_qns, 2)}%)"
benchmark_df.loc["Lexical", "BigCodeBench Accuracy"] = f"{bigcodebench_lexical_total_success}/{bigcodebench_lexical_total_answered} ({round(bigcodebench_lexical_total_success*100/bigcodebench_lexical_total_answered, 2)}%)"
benchmark_df.loc["Aggregated Results", "BigCodeBench Inconsistencies"] = f"{bigcodebench_lexical_total_inc}/{bigcodebench_lexical_total_qns} ({round(bigcodebench_lexical_total_inc*100/bigcodebench_lexical_total_qns, 2)}%)"
benchmark_df.loc["Aggregated Results", "BigCodeBench Accuracy"] = f"{bigcodebench_lexical_total_success}/{bigcodebench_lexical_total_answered} ({round(bigcodebench_lexical_total_success*100/bigcodebench_lexical_total_answered, 2)}%)"

all_cat_dict['Lexical']['total_inconsistencies'] += bigcodebench_lexical_total_inc
all_cat_dict['Lexical']['total_questions'] += bigcodebench_lexical_total_qns
all_cat_dict['Lexical']['total_success'] += bigcodebench_lexical_total_success
all_cat_dict['Lexical']['total_answered'] += bigcodebench_lexical_total_answered

# End BCB results aggregation #

for mut_cat, res_dict in all_cat_dict.items():
    mut_inconsistencies = res_dict['total_inconsistencies']
    mut_questions = res_dict['total_questions']
    mut_success = res_dict['total_success']
    mut_answered = res_dict['total_answered']

    benchmark_df.loc[mut_cat, "Aggregated Mutation Inc."] =  f"{mut_inconsistencies}/{mut_questions} ({round(mut_inconsistencies*100/mut_questions, 2)}%)"
    benchmark_df.loc[mut_cat, "Aggregated Mutation Acc."] =  f"{mut_success}/{mut_answered} ({round(mut_success*100/mut_answered, 2)}%)"
    

print(benchmark_df.to_string())
benchmark_df.to_csv('benchmark.csv')

                   CodeMMLU Inconsistencies   CodeMMLU Accuracy HumanEval Inconsistencies    HumanEval Accuracy CruxEval Inconsistencies     CruxEval Accuracy BigCodeBench Inconsistencies BigCodeBench Accuracy Aggregated Mutation Inc. Aggregated Mutation Acc.
Logical                    743/1930 (38.5%)   905/1947 (46.48%)        3195/32116 (9.95%)  23512/32094 (73.26%)      2321/10962 (21.17%)   6638/10959 (60.57%)                          NaN                   NaN      6259/45008 (13.91%)     31055/45000 (69.01%)
Syntactic                  139/820 (16.95%)     556/820 (67.8%)        1134/13580 (8.35%)  10173/13574 (74.94%)       1418/8820 (16.08%)    5883/8815 (66.74%)                          NaN                   NaN      2691/23220 (11.59%)     16612/23209 (71.58%)
Lexical                   218/2172 (10.04%)  1702/2172 (78.36%)       3832/34860 (10.99%)  25973/34841 (74.55%)      4737/27188 (17.42%)  18051/27182 (66.41%)          4223/15487 (27.27%)   8699/15276 (56.95%)     13010/

# MuCoCo Results Aggregated Across Tasks (Table VII)

In [28]:
import copy

task_df = pd.DataFrame()
all_cat_dict = {}


for task, mut_cat_dict in task_dict.items():

    benchmark_inconsistencies = 0
    benchmark_questions = 0
    benchmark_success = 0
    benchmark_answered = 0
    benchmark_distance = 0

    for mut_cat, res_dir in mut_cat_dict.items():
        if not res_dir:
            continue

        # Make a defensive copy so we don’t mutate shared references
        res_dir = copy.deepcopy(res_dir)

        mut_inconsistencies = res_dir['total_inconsistencies']
        mut_questions = res_dir['total_questions']
        mut_success = res_dir['total_success']
        mut_answered = res_dir['total_answered']
        mut_cum_inc_dist = res_dir['cumulative_inconsistency_distance']

        if task == "code_generation" and mut_cat == "Lexical":
            mut_inconsistencies += bigcodebench_lexical_total_inc
            mut_questions += bigcodebench_lexical_total_qns
            mut_success += bigcodebench_lexical_total_success
            mut_answered += bigcodebench_lexical_total_answered
            mut_cum_inc_dist += bigcodebench_lexical_total_inc_dist

        task_df.loc[mut_cat, f"{task} Inconsistencies"] = f"{mut_inconsistencies}/{mut_questions} ({round(mut_inconsistencies*100/mut_questions, 2)}%)"
        if task == "code_generation":
            task_df.loc[mut_cat, f"{task} Inc. Distance"] = f"{mut_cum_inc_dist}/{mut_questions} ({round(mut_cum_inc_dist/mut_questions, 5)})"
        task_df.loc[mut_cat, f"{task} Accuracy"] = f"{mut_success}/{mut_answered} ({round(mut_success*100/mut_answered, 2)}%)"

        benchmark_inconsistencies += mut_inconsistencies
        benchmark_questions += mut_questions
        benchmark_success += mut_success
        benchmark_answered += mut_answered
        benchmark_distance += mut_cum_inc_dist

        d = all_cat_dict.get(mut_cat, {})
        if not d:
            all_cat_dict[mut_cat] = copy.deepcopy(res_dir)
        else:
            for key, value in d.items():
                d[key] += res_dir[key]
            all_cat_dict[mut_cat] = d


    task_df.loc["Aggregated Results", f"{task} Inconsistencies"] = f"{benchmark_inconsistencies}/{benchmark_questions} ({round(benchmark_inconsistencies*100/benchmark_questions, 2)}%)"
    if task == 'code_generation':
        task_df.loc["Aggregated Results", f"{task} Inc. Distance"] = f"{benchmark_distance}/{benchmark_questions} ({round(benchmark_distance/benchmark_questions, 5)})"
    task_df.loc["Aggregated Results", f"{task} Accuracy"] = f"{benchmark_success}/{benchmark_answered} ({round(benchmark_success*100/benchmark_answered, 2)}%)"


for mut_cat, res_dict in all_cat_dict.items():
    mut_inconsistencies = res_dict['total_inconsistencies']
    mut_questions = res_dict['total_questions']
    mut_success = res_dict['total_success']
    mut_answered = res_dict['total_answered']

    if mut_cat == "Lexical":
        mut_inconsistencies += bigcodebench_lexical_total_inc
        mut_questions += bigcodebench_lexical_total_qns
        mut_success += bigcodebench_lexical_total_success
        mut_answered += bigcodebench_lexical_total_answered
        mut_cum_inc_dist += bigcodebench_lexical_total_inc_dist

    task_df.loc[mut_cat, "Aggregated Mutation Inc."] =  f"{mut_inconsistencies}/{mut_questions} ({round(mut_inconsistencies*100/mut_questions, 2)})"
    task_df.loc[mut_cat, "Aggregated Mutation Acc."] =  f"{mut_success}/{mut_answered} ({round(mut_success*100/mut_answered, 2)})"
    

print(task_df.to_string())

                   mcq_inconsistency Inconsistencies mcq_inconsistency Accuracy input_prediction Inconsistencies input_prediction Accuracy output_prediction Inconsistencies output_prediction Accuracy code_generation Inconsistencies      code_generation Inc. Distance code_generation Accuracy Aggregated Mutation Inc. Aggregated Mutation Acc.
Logical                             743/1930 (38.5%)          905/1947 (46.48%)               1526/21539 (7.08%)       16342/21532 (75.9%)               3990/21539 (18.52%)       13808/21521 (64.16%)                             NaN                                NaN                      NaN       6259/45008 (13.91)      31055/45000 (69.01)
Syntactic                           139/820 (16.95%)            556/820 (67.8%)                 661/11200 (5.9%)       9017/11197 (80.53%)               1891/11200 (16.88%)        7039/11192 (62.89%)                             NaN                                NaN                      NaN       2691/23220 (11.59)

# Aggregating MuCoCo results across models (Table VI)

In [29]:
import pandas as pd

data = overall_dict
categories = ["Logical", "Syntactic", "Lexical"]
models = list(data.keys())

rows = []

# keep track of total inconsistencies per model
model_inconsistency_totals = {model: 0 for model in models}

for cat in categories:
    row = {"Category": cat}
    cat_inconsistency = 0
    cat_total_questions = 0
    cat_total_success = 0
    cat_total_answered = 0

    for model in models:
        vals = data[model][cat]

        bigcodebench_inconsistencies = bigcodebench_res[model]["Lexical"]

        model_inconsistencies = vals['total_inconsistencies'] 
        model_questions = vals['total_questions']
        model_successes = vals['total_success'] 
        model_answered = vals['total_answered']
    

        if cat == "Lexical":
            model_inconsistencies += bigcodebench_inconsistencies['total_inconsistencies']
            model_questions += bigcodebench_inconsistencies['total_questions']
            model_successes += bigcodebench_inconsistencies['total_success']
            model_answered += bigcodebench_inconsistencies['total_answered']

        # string representation for reporting
        inc = f'{model_inconsistencies}/{model_questions} = {round(model_inconsistencies / model_questions * 100, 2)}'
        acc = f'{model_successes}/{model_answered} = {round(model_successes / model_answered * 100, 2)}'
            
        row[f"{model} Inconsistency"] = inc
        row[f"{model} Accuracy"] = acc

        # accumulate for averages
        if "ensemble" not in model:
            cat_inconsistency += model_inconsistencies
            cat_total_questions += model_questions
            cat_total_success += model_successes
            cat_total_answered += model_answered

        # accumulate for global weightage
        model_inconsistency_totals[model] += vals["total_inconsistencies"]

    # per-category average
    row["Average Inconsistency"] = f"{cat_inconsistency}/{cat_total_questions} = {round(cat_inconsistency*100 / cat_total_questions, 2)}"
    row["Average Accuracy"] = f"{cat_total_success}/{cat_total_answered} = {round(cat_total_success*100 / cat_total_answered, 2)}"
    rows.append(row)

avg_row = {"Category": "All"}
sums = {col: {"num": 0, "den": 0} for col in rows[0].keys() if col != "Category"}

for row in rows:
    for col in sums.keys():
        val = row[col]
        if isinstance(val, str) and "/" in val:
            try:
                frac_part = val.split('=')[0].strip()
                num, den = frac_part.split('/')
                num, den = int(num.strip()), int(den.strip())
                sums[col]["num"] += num
                sums[col]["den"] += den
            except Exception:
                continue

for col, vals in sums.items():
    num, den = vals["num"], vals["den"]
    if den > 0:
        avg_row[col] = f"{num}/{den} = {round(num * 100 / den, 2)}"
    else:
        avg_row[col] = "0/0 = 0.0"

rows.append(avg_row)

df = pd.DataFrame(rows)
print(df.to_string())
df.to_csv('models.csv')


    Category Qwen2.5-Coder-14B-Instruct Inconsistency Qwen2.5-Coder-14B-Instruct Accuracy Gemma-3-12b-it Inconsistency Gemma-3-12b-it Accuracy DeepSeek-V3.2-Exp (Non-thinking Mode) Inconsistency DeepSeek-V3.2-Exp (Non-thinking Mode) Accuracy LLama-3.1-8B Inconsistency LLama-3.1-8B Accuracy GPT-5 Inconsistency       GPT-5 Accuracy GPT-4o Inconsistency      GPT-4o Accuracy codestral-2508 Inconsistency codestral-2508 Accuracy Average Inconsistency       Average Accuracy
0    Logical                          626/6424 = 9.74                   4904/6433 = 76.23             997/6424 = 15.52       4254/6433 = 66.13                                   1121/6432 = 17.43                               4050/6429 = 63.0          1159/6432 = 18.02     3208/6432 = 49.88     123/6432 = 1.91    6255/6414 = 97.52    1253/6432 = 19.48    4130/6428 = 64.25             980/6432 = 15.24       4254/6431 = 66.15    6259/45008 = 13.91    31055/45000 = 69.01
1  Syntactic                          223/3318 = 6.72   

# Formulation of Model Weights used for weighted model ensemble

In [30]:

import numpy as np
valid_models = [m for m in models if "ensemble" not in m]

raw = np.array([model_inconsistency_totals[m] for m in valid_models], dtype=float)

inv = 1 / raw

weights = inv / np.sum(inv)

df_weights = pd.DataFrame({
    "Model": valid_models,
    "Inverse Weight": np.round(weights, 6)
}).sort_values(by="Inverse Weight", ascending=False).reset_index(drop=True)

print("Sum of weights:", np.sum(df_weights["Inverse Weight"]))
print(df_weights.to_string())

Sum of weights: 1.0
                                   Model  Inverse Weight
0                                  GPT-5        0.592421
1             Qwen2.5-Coder-14B-Instruct        0.102790
2                         codestral-2508        0.071325
3                         Gemma-3-12b-it        0.064358
4  DeepSeek-V3.2-Exp (Non-thinking Mode)        0.061877
5                                 GPT-4o        0.054403
6                           LLama-3.1-8B        0.052826
